# Gamma-Theta Coupling in Hippocampus

Pipeline following Schomburg et al. 2014 and Sirota et al. 2008.

Goal: find gamma bursts in CA1 stratum radiatum (rad, CA3 input), stratum
lacunosum-moleculare (lm, entorhinal input) and the local CA1 pyramidal
layer (ca1Pyr, fast gamma), and check at which theta phase each of these
gamma sources is active.

Steps:
1. load LFP, keep only "run" epochs
2. whiten each channel (removes 1/f, flattens spectrum before ICA)
3. bandpass to gamma, PCA + ICA separately per shank
4. use CSD (current source density) on the ICA loading to find which
   anatomical layer each component comes from
5. match the "same" component across shanks with coherence, fix sign
   flips with cross-spectrum
6. detect gamma bursts (2D spectrogram local maxima)
7. get theta phase from raw (non-whitened) LFP
8. circular statistics: is each layer's gamma locked to a theta phase?


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
from scipy.stats import kurtosis
from sklearn.decomposition import PCA, FastICA

import wp7_helpers as wp7

%matplotlib inline


## 1. Load data and restrict to "run" epochs

Only keep the parts of the recording where the animal was actually running,
since that is when theta is strong and clean.


In [ ]:
sub = 'XXX'   # fill in subject id
ses = 'XXX'   # fill in session id

lfp, fs = wp7.load_lfp(sub, ses)   # lfp shape: (n_samples, n_channels)

channels = pd.read_csv(f'{sub}_{ses}_channels.tsv', sep='\t')
events = pd.read_csv(f'{sub}_{ses}_events.tsv', sep='\t')

# channel 96 is the auxiliary/MISC channel, not a real recording channel
channels = channels.iloc[:96]
channels['region'] = channels['region'].str.strip()

run_epochs = events[events['trial_type'] == 'run']
print(run_epochs.shape)


In [ ]:
# concatenate only the run segments, first 96 channels only
lfp96 = np.concatenate([lfp[r.s0:r.s1, :96] for _, r in run_epochs.iterrows()])
print(lfp96.shape)


In [ ]:
# map region name -> channel indices, and the reverse lookup
layers = {}
for region in channels['region'].unique():
    layers[region] = channels.index[channels['region'] == region].to_numpy()

ch_to_layer = {}
for region, chs in layers.items():
    for ch in chs:
        ch_to_layer[ch] = region

layers


## 2. AR whitening

- Raw LFP has a 1/f spectrum. ICA works better if the input is "white" (flat spectrum, no autocorrelation) 
- fit a short autoregressive model per channel and keep only the residual


In [ ]:
ar_order = 3
lfp_white = np.zeros_like(lfp96)

for ch in range(lfp96.shape[1]):
    lfp_white[:, ch] = wp7.whiten_ar(lfp96[:, ch], order=ar_order)

# drop the first ar_order samples (not defined after whitening)
lfp_white = lfp_white[ar_order:, :]
print(lfp_white.shape)


## 3. Gamma bandpass + PCA + ICA, per shank

ICA separately per shank (16 channels each), not on all 96 channels
together:
- CSD needs physically adjacent, equally spaced channels. That is only
  true within one shank, not across shanks.
- doing all 96 channels together would bias ICA toward whatever source is
  coherent across the whole probe, not local layer-specific sources.

Before ICA we run PCA
- reduces dimensionality (16 channels -> 6 components). I picked 6 by
  looking at the explained-variance curve: it elbows around 3-4 components,
  and I kept a bit of headroom for DG/CA3 contamination and artifacts.
- whitens the data, so FastICA only has to find a rotation that maximizes
  non-Gaussianity (logcosh contrast function), instead of a full general
  unmixing problem.


In [ ]:
gamma_low, gamma_high = 30, 200
sos = signal.butter(4, [gamma_low, gamma_high], btype='bandpass', fs=fs, output='sos')

n_shanks = 6
ch_per_shank = 16
n_components = 6

ICA_shank_S = []   # ICA component time series, one array per shank
ICA_shank_A = []   # mixing matrix (channels x components), one per shank

for shank in range(n_shanks):
    ch_idx = np.arange(shank * ch_per_shank, (shank + 1) * ch_per_shank)
    shank_data = lfp_white[:, ch_idx]

    shank_gamma = signal.sosfiltfilt(sos, shank_data, axis=0)

    pca = PCA(n_components=n_components, whiten=True)
    pca_scores = pca.fit_transform(shank_gamma)

    ica = FastICA(whiten=False, max_iter=1000)
    S = ica.fit_transform(pca_scores)          # (n_samples, n_components)
    A_pca = ica.mixing_                        # (n_components, n_components)

    # back-project mixing matrix into the original 16-channel space
    A_original = pca.components_.T @ A_pca     # (16, n_components)

    ICA_shank_S.append(S)
    ICA_shank_A.append(A_original)

    print(f'shank {shank}: explained variance {pca.explained_variance_ratio_.sum():.2f}')


In [ ]:
# explained variance elbow plot, just to document the choice of 6
pca_full = PCA(n_components=16)
pca_full.fit(lfp_white[:, :16])

plt.figure()
plt.plot(np.arange(1, 17), pca_full.explained_variance_ratio_, 'o-')
plt.xlabel('component')
plt.ylabel('explained variance ratio')
plt.title('PCA explained variance, shank 0')
plt.show()


## 4. CSD to identify anatomical layer of each component

CSD is the second spatial derivative of the ICA loading profile along the
shank. A sink (negative/blue) marks where synaptic input is arriving - so
a sink tells us the anatomical layer that component's source lives in.

Exception: the ca1Pyr component shows up as a SOURCE, not a sink, because
of perisomatic inhibition at the pyramidal layer.


In [ ]:
def compute_csd_loading(loading_vector, smooth_sigma=2.0):
    from scipy.ndimage import gaussian_filter1d
    norm = loading_vector / np.max(np.abs(loading_vector))
    smoothed = gaussian_filter1d(norm, sigma=smooth_sigma)
    csd = -np.gradient(np.gradient(smoothed))
    return csd


In [ ]:
# plot CSD profile for every component of one shank, to assign layers by eye
shank = 2   # 0-indexed, this is probe shank 3

plt.figure(figsize=(10, 6))
for comp in range(n_components):
    loading = ICA_shank_A[shank][:, comp]
    csd = compute_csd_loading(loading)
    plt.subplot(2, 3, comp + 1)
    plt.plot(csd, np.arange(16), 'o-')
    plt.gca().invert_yaxis()
    plt.title(f'IC {comp}')
    plt.xlabel('CSD')
    plt.ylabel('channel (depth)')
plt.tight_layout()
plt.show()


## 5. Kurtosis-based artifact rejection

Components that are dominated by a single large artifact (movement, chewing,
etc) tend to have very high kurtosis (sharp, peaky distribution instead of
roughly Gaussian/sub-Gaussian gamma activity). We flag and drop those.


In [ ]:
kurtosis_threshold = 5.0

good_components = {}   # shank -> list of component indices kept

for shank in range(n_shanks):
    k = kurtosis(ICA_shank_S[shank], axis=0)
    keep = np.where(np.abs(k) < kurtosis_threshold)[0]
    good_components[shank] = keep
    print(f'shank {shank}: kurtosis {np.round(k, 2)}, keeping {keep}')


## 6. Match components across shanks

### Why coherence and not correlation
Correlation only works for signals aligned sample-by-sample, in time. Two
copies of the "same" gamma source on different shanks can have a small
time/phase shift and still be the same underlying source. Coherence
measures similarity frequency-by-frequency and is not thrown off by that
kind of lag, so it is a better match criterion for a source that is
volume-conducted / shared across shanks with some delay.

### Why cross-spectrum after coherence
Coherence is sign-blind (it is a magnitude, from a squared/abs quantity),
so it will happily call a component and its sign-flipped mirror image a
perfect match. To find out whether an actual flip occurred, we need phase
information, which is what the (complex-valued) cross-spectrum gives us.
A phase difference near 0 deg means they're in phase (no flip needed), near
180 deg means we should multiply one of them by -1.


In [ ]:
# only use shanks 1,2,3 (shanks 2,3,4) which are the shanks of interest
shanks_to_use = [1, 2, 3]

all_signals = []
all_labels = []       # (shank, component) for each entry in all_signals

for shank in shanks_to_use:
    for comp in good_components[shank]:
        all_signals.append(ICA_shank_S[shank][:, comp])
        all_labels.append((shank, comp))

n_all = len(all_signals)
print(n_all, 'components total')


In [ ]:
# coherence matrix across all kept components, on a short subset
# (full coherence on the whole recording takes way too long)
max_samples = int(5 * 60 * fs)

coh_matrix = np.zeros((n_all, n_all))

for i in range(n_all):
    for j in range(i, n_all):
        if i == j:
            coh_matrix[i, j] = 1.0
            continue
        f, Cxy = wp7.coherence(all_signals[i][:max_samples],
                                all_signals[j][:max_samples],
                                fs=fs, nperseg=int(2 * fs))
        gamma_mask = (f >= gamma_low) & (f <= 100)
        coh_matrix[i, j] = np.mean(Cxy[gamma_mask])
        coh_matrix[j, i] = coh_matrix[i, j]

print(coh_matrix.shape)


In [ ]:
# for every component on a reference shank, find its best match on every
# other shank -> this gives one "group" per reference component
reference_shank = 2   # shank index into shanks_to_use list meaning, see below

matched_groups = []

ref_idx_list = [i for i, (sh, c) in enumerate(all_labels) if sh == shanks_to_use[1]]

for ref_i in ref_idx_list:
    group = [all_labels[ref_i]]
    for shank in shanks_to_use:
        if shank == all_labels[ref_i][0]:
            continue
        candidates = [i for i, (sh, c) in enumerate(all_labels) if sh == shank]
        best = max(candidates, key=lambda i: coh_matrix[ref_i, i])
        group.append(all_labels[best])
    matched_groups.append(group)

for g in matched_groups:
    print(g)


In [ ]:
# cross-spectrum sign check: does a component on another shank need flipping
# to be in phase with the reference component in the same group?
sign_flips = {}   # (shank, comp) -> +1 or -1

for group in matched_groups:
    ref_shank, ref_comp = group[0]
    sig_ref = ICA_shank_S[ref_shank][:, ref_comp]

    for shank, comp in group:
        sig = ICA_shank_S[shank][:, comp]
        f, Cxy = wp7.cross_spectrum(sig_ref, sig, fs=fs, nperseg=int(2 * fs))
        gamma_mask = (f >= gamma_low) & (f <= 100)
        mean_phase_deg = np.degrees(np.mean(np.angle(Cxy[gamma_mask])))

        flip = -1 if abs(mean_phase_deg) > 90 else 1
        sign_flips[(shank, comp)] = flip
        print(shank, comp, 'phase', round(mean_phase_deg, 1), 'flip', flip)


In [ ]:
# apply the sign flips once, in place, to both the time series and the
# CSD-relevant loading vectors
for (shank, comp), flip in sign_flips.items():
    ICA_shank_S[shank][:, comp] *= flip
    ICA_shank_A[shank][:, comp] *= flip


## 7. Final layer assignment

Looking at the CSD profiles of the matched groups by eye: group 2 sinks in
stratum radiatum, group 5 sinks in stratum lacunosum-moleculare, group 3 is
a source at the pyramidal layer (ca1Pyr).


In [ ]:
groups = {
    'rad':    matched_groups[2],
    'lm':     matched_groups[5],
    'ca1Pyr': matched_groups[3],
}
groups


In [ ]:
# plot CSD of the matched components for each group, one row per layer
plt.figure(figsize=(10, 8))
plot_i = 1
for name, group in groups.items():
    for shank, comp in group:
        loading = ICA_shank_A[shank][:, comp]
        csd = compute_csd_loading(loading)
        plt.subplot(3, 3, plot_i)
        plt.plot(csd, np.arange(16), 'o-')
        plt.gca().invert_yaxis()
        plt.title(f'{name}, shank {shank}')
        plot_i += 1
plt.tight_layout()
plt.show()


## 8. Gamma burst detection

Method: 2D local maxima on the spectrogram (time x frequency), not a simple
1D Hilbert-envelope threshold. A burst is a local peak in time AND
frequency at once.

- gentle smoothing of the spectrogram before peak finding
  (`window_size=0.7`), heavy smoothing (`window_size=3`) blurs individual
  bursts into one continuous blob
- threshold = 90th percentile of the (dB) spectrogram
- after `find_peaks_in_spectrogram_fast`, apply a minimum time separation
  of 50 ms (about one gamma cycle) between detected peaks, otherwise many
  small local maxima get detected inside one obvious burst


In [ ]:
GAMMA_LOW, GAMMA_HIGH = 30, 200
min_sep_sec = 0.050

window_sec = 0.1
overlap = 0.9
nw = 2
nfft = 2**14   # zero-padded for a finer frequency grid (no real added resolution,
               # but avoids gappy/discrete-looking burst frequency histograms)

results_full = {}

for name, group in groups.items():
    results_full[name] = {}
    for i, (shank, comp) in enumerate(group):
        sig = ICA_shank_S[shank][:, comp]

        times_sec, freqs_axis, S = wp7.spectrogram_multitaper(
            sig, fs=fs, window_sec=window_sec, overlap=overlap, nw=nw, nfft=nfft)

        freq_mask = (freqs_axis >= GAMMA_LOW) & (freqs_axis <= GAMMA_HIGH)
        S = S[freq_mask, :]
        freqs_axis_g = freqs_axis[freq_mask]

        S_db = 10 * np.log10(S)
        S_smooth = wp7.smooth_spectrogram(S_db, window_size=0.7)

        threshold = np.percentile(S_smooth, 90)
        peaks_t_idx, peaks_f_idx = wp7.find_peaks_in_spectrogram_fast(S_smooth, threshold)

        peak_times = times_sec[peaks_t_idx]
        peak_freqs = freqs_axis_g[peaks_f_idx]

        # minimum time separation filter (keep earliest of any pair closer than min_sep_sec)
        order = np.argsort(peak_times)
        peak_times, peak_freqs = peak_times[order], peak_freqs[order]
        keep = [0]
        for k in range(1, len(peak_times)):
            if peak_times[k] - peak_times[keep[-1]] > min_sep_sec:
                keep.append(k)
        peak_times = peak_times[keep]
        peak_freqs = peak_freqs[keep]

        results_full[name][i] = {
            'times_sec': peak_times,
            'freqs_hz': peak_freqs,
        }
        print(name, i, 'n bursts:', len(peak_times))


In [ ]:
# sanity check plot: spectrogram + detected bursts, for one component
name, i = 'rad', 0
shank, comp = groups[name][i]
sig = ICA_shank_S[shank][:, comp]

t0, t1 = 0, 10   # seconds, zoom window
s0, s1 = int(t0 * fs), int(t1 * fs)

times_sec, freqs_axis, S = wp7.spectrogram_multitaper(
    sig[s0:s1], fs=fs, window_sec=window_sec, overlap=overlap, nw=nw, nfft=nfft)
freq_mask = (freqs_axis >= GAMMA_LOW) & (freqs_axis <= GAMMA_HIGH)
S_db = 10 * np.log10(S[freq_mask, :])

plt.figure(figsize=(10, 4))
plt.imshow(S_db, aspect='auto', origin='lower',
           extent=[t0, t1, freqs_axis[freq_mask][0], freqs_axis[freq_mask][-1]])

in_window = (results_full[name][i]['times_sec'] >= t0) & (results_full[name][i]['times_sec'] < t1)
plt.scatter(results_full[name][i]['times_sec'][in_window],
            results_full[name][i]['freqs_hz'][in_window],
            facecolors='none', edgecolors='white')

plt.xlabel('time (s)')
plt.ylabel('frequency (Hz)')
plt.title(f'{name} component {i}, detected bursts')
plt.show()


## 9. Theta phase

Theta phase comes from the RAW (not whitened, not gamma-filtered) LFP, from
a CA1 pyramidal layer channel. We use the Hilbert transform (bandpass 6-12
Hz + Hilbert) to get instantaneous phase.


In [ ]:
theta_ch = layers['ca1Pyr'][0]
theta_ref = lfp96[ar_order:, theta_ch]   # same time alignment as lfp_white

z = wp7.analytic_signal(theta_ref, fs, band=(6, 12))
theta_phase = np.angle(z)


In [ ]:
# check: is the phase distribution roughly uniform (no artifact bias)?
plt.figure()
plt.hist(theta_phase, bins=36)
plt.xlabel('theta phase (rad)')
plt.ylabel('count')
plt.title('theta phase distribution')
plt.show()


## 10. Circular statistics: gamma burst timing vs theta phase

For each burst, look up the theta phase at that moment. Then compute:
- R (resultant vector length / PLV) - inflates with n, use with care
- PPC (pairwise phase consistency) - unbiased alternative to R
- Rayleigh test - is the phase distribution significantly non-uniform?
- circular mean - the preferred theta phase


In [ ]:
for name, group in groups.items():
    for i in results_full[name]:
        peak_times = results_full[name][i]['times_sec']
        peak_idx = (peak_times * fs).astype(int)
        peak_idx = peak_idx[peak_idx < len(theta_phase)]
        phases = theta_phase[peak_idx]

        R = wp7.circ_r(phases)
        mean_phase = wp7.circ_mean(phases)
        z_val, p_val = wp7.rayleigh(phases)
        ppc_val = wp7.ppc(phases)

        print(f'{name} IC{i}: n={len(phases)}  R={R:.3f}  PPC={ppc_val:.3f}  '
              f'mean_phase={np.degrees(mean_phase):.1f} deg  p={p_val:.4g}')


In [ ]:
# circular histogram, 3x3 grid: rows = layers, columns = matched components
plt.figure(figsize=(10, 10))
plot_i = 1
for name, group in groups.items():
    for i in results_full[name]:
        peak_times = results_full[name][i]['times_sec']
        peak_idx = (peak_times * fs).astype(int)
        peak_idx = peak_idx[peak_idx < len(theta_phase)]
        phases = theta_phase[peak_idx]

        plt.subplot(3, 3, plot_i, projection='polar')
        plt.hist(phases, bins=18)
        plt.title(f'{name} IC{i}')
        plot_i += 1
plt.tight_layout()
plt.show()


## 11. Slow / medium / fast gamma sub-bands

Same circular analysis, but split bursts by their peak frequency into
slow (30-50 Hz, CA3-like), medium (50-90 Hz, EC-like) and fast (90-150 Hz,
local CA1) gamma, to see if the theta-phase preference depends on gamma
sub-band.


In [ ]:
gamma_bands = {
    'slow (30-50 Hz)': (30, 50),
    'medium (50-90 Hz)': (50, 90),
    'fast (90-150 Hz)': (90, 150),
}

for band_name, (f_lo, f_hi) in gamma_bands.items():
    plt.figure(figsize=(10, 10))
    plt.suptitle(band_name)
    plot_i = 1
    for name, group in groups.items():
        for i in results_full[name]:
            peak_times = results_full[name][i]['times_sec']
            peak_freqs = results_full[name][i]['freqs_hz']

            band_mask = (peak_freqs >= f_lo) & (peak_freqs < f_hi)
            peak_idx = (peak_times[band_mask] * fs).astype(int)
            peak_idx = peak_idx[peak_idx < len(theta_phase)]
            phases = theta_phase[peak_idx]

            plt.subplot(3, 3, plot_i, projection='polar')
            plt.hist(phases, bins=18)
            plt.title(f'{name} IC{i}, n={len(phases)}')
            plot_i += 1
    plt.tight_layout()
    plt.show()


## 12. Threshold / frequency-window robustness check

Is the main result (rad vs lm theta-phase separation) just an artifact of
the particular threshold (90th percentile) we picked? Sweep the detection
threshold, and within each threshold slide a 20 Hz frequency window across
30-200 Hz (keeping only bins with at least 20 bursts), and track R and
preferred phase as a function of frequency.


In [ ]:
thresholds_to_try = [50, 60, 70, 80, 90]
freq_window_hz = 20.0
min_bursts = 20

coupling_vs_freq_by_threshold = {}

for thresh_pct in thresholds_to_try:
    coupling_vs_freq_by_threshold[thresh_pct] = {}

    for name, group in groups.items():
        center_freqs = []
        R_curve = []

        for shank, comp in group:
            sig = ICA_shank_S[shank][:, comp]

            times_sec, freqs_axis, S = wp7.spectrogram_multitaper(
                sig, fs=fs, window_sec=window_sec, overlap=overlap, nw=nw, nfft=nfft)
            freq_mask = (freqs_axis >= GAMMA_LOW) & (freqs_axis <= GAMMA_HIGH)
            S_db = 10 * np.log10(S[freq_mask, :])
            freqs_axis_g = freqs_axis[freq_mask]

            S_smooth = wp7.smooth_spectrogram(S_db, window_size=0.7)
            threshold = np.percentile(S_smooth, thresh_pct)
            peaks_t_idx, peaks_f_idx = wp7.find_peaks_in_spectrogram_fast(S_smooth, threshold)

            peak_times = times_sec[peaks_t_idx]
            peak_freqs = freqs_axis_g[peaks_f_idx]

            for f_center in np.arange(GAMMA_LOW, GAMMA_HIGH, 5):
                band_mask = (peak_freqs >= f_center - freq_window_hz / 2) & \
                            (peak_freqs < f_center + freq_window_hz / 2)
                if band_mask.sum() < min_bursts:
                    continue
                peak_idx = (peak_times[band_mask] * fs).astype(int)
                peak_idx = peak_idx[peak_idx < len(theta_phase)]
                phases = theta_phase[peak_idx]

                center_freqs.append(f_center)
                R_curve.append(wp7.circ_r(phases))

        coupling_vs_freq_by_threshold[thresh_pct][name] = {
            'center_freqs': np.array(center_freqs),
            'R_curve': np.array(R_curve),
        }


In [ ]:
# one panel per layer, one line per threshold
plt.figure(figsize=(12, 4))
for plot_i, name in enumerate(groups.keys(), start=1):
    plt.subplot(1, 3, plot_i)
    for thresh_pct in thresholds_to_try:
        d = coupling_vs_freq_by_threshold[thresh_pct][name]
        plt.plot(d['center_freqs'], d['R_curve'], label=f'{thresh_pct}th pct')
    plt.xlabel('gamma frequency (Hz)')
    plt.ylabel('R')
    plt.title(name)
    plt.legend(fontsize=7)
plt.tight_layout()
plt.show()


In [ ]:
# summary: preferred phase and R vs threshold, averaged over frequency, per layer
plt.figure(figsize=(12, 4))
for plot_i, name in enumerate(groups.keys(), start=1):
    mean_R = [coupling_vs_freq_by_threshold[t][name]['R_curve'].mean()
              for t in thresholds_to_try]
    plt.subplot(1, 3, plot_i)
    plt.plot(thresholds_to_try, mean_R, 'o-')
    plt.xlabel('threshold percentile')
    plt.ylabel('mean R across freq')
    plt.title(name)
plt.tight_layout()
plt.show()


## Main result

- rad (CA3 input) and lm (entorhinal input) prefer theta phases roughly
  90-100 deg apart -> consistent with Schomburg 2014: CA3 and entorhinal
  input to CA1 are temporally segregated by theta phase.
- ca1Pyr (local fast gamma) shows weaker / less consistent phase locking
  across the robustness sweep, since it is generated locally rather than
  driven by a phase-locked afferent input.
